Библиотеки для кода: 

In [104]:

import time
import os
import re
import numpy as np
import pandas as pd
from datetime import datetime

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import hstack

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")


from sklearn.model_selection import GridSearchCV 
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer


from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier


from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel, BertTokenizer, BertModel
import warnings
warnings.filterwarnings('ignore')



import requests
from PIL import Image
from io import BytesIO


pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)



Загружаем данные 

In [105]:


downloads_path = r'C:\Users\Elena\Downloads'


train = pd.read_csv(os.path.join(downloads_path, 'trainКР.csv'))
test = pd.read_csv(os.path.join(downloads_path, 'testКР.csv'))
sample = pd.read_csv(os.path.join(downloads_path, 'sample_submission.csv'))


print("РАЗМЕРЫ ДАТАСЕТОВ")

print(f"Train: {train.shape[0]} строк, {train.shape[1]} колонок")
print(f"Test:  {test.shape[0]} строк, {test.shape[1]} колонок")
print(f"Sample: {sample.shape[0]} строк, {sample.shape[1]} колонок")


print("КОЛОНКИ В TRAIN")

print(train.columns.tolist())


print("ЦЕЛЕВЫЕ КЛАССЫ")

print(train['category_1'].value_counts())


print("ПРОПУСКИ В TRAIN")

print(train.isnull().sum())


print("ПЕРВЫЕ 3 СТРОКИ")

print(train.head(3))


РАЗМЕРЫ ДАТАСЕТОВ
Train: 9632 строк, 11 колонок
Test:  2409 строк, 10 колонок
Sample: 191 строк, 2 колонок
КОЛОНКИ В TRAIN
['id', 'category_1', 'title', 'product_rating', 'selling_price', 'mrp', 'seller_name', 'seller_rating', 'description', 'highlights', 'image_links']
ЦЕЛЕВЫЕ КЛАССЫ
category_1
Women's wear              1937
Men's wear                1888
Bady and Kids             1870
Home and Furniture        1696
Sports, Books and More    1456
Electronics                785
Name: count, dtype: int64
ПРОПУСКИ В TRAIN
id                   0
category_1           0
title               13
product_rating      61
selling_price       21
mrp                304
seller_name        169
seller_rating      168
description       5609
highlights        4392
image_links          0
dtype: int64
ПЕРВЫЕ 3 СТРОКИ
     id    category_1                                              title  \
0  2126   Electronics  Flipkart SmartBuy FOLD Mobile Stand Holder - [...   
1  3403    Men's wear  TRYBEST Men Tailo

данные содержат текстовые (title, description, highlights) и числовые (рейтинги, цены) признаки. 
Целевая переменная имеет 6 классов с дисбалансом (Electronics меньше всего), поэтому используем macro F1-score.

Делаем анализ и очистку наших данных 

In [106]:

# 1. Очистка цен (удаляем символы валют и запятые)
def clean_price(price_str):
    if pd.isna(price_str):
        return np.nan
    cleaned = re.sub(r'[^\d.]', '', str(price_str))
    try:
        return float(cleaned)
    except:
        return np.nan

train['selling_price'] = train['selling_price'].apply(clean_price)
train['mrp'] = train['mrp'].apply(clean_price)
test['selling_price'] = test['selling_price'].apply(clean_price)
test['mrp'] = test['mrp'].apply(clean_price)

# Создаем признак скидки
train['discount'] = (train['mrp'] - train['selling_price']) / train['mrp']
test['discount'] = (test['mrp'] - test['selling_price']) / test['mrp']
train['discount'] = train['discount'].replace([np.inf, -np.inf], 0).fillna(0)
test['discount'] = test['discount'].replace([np.inf, -np.inf], 0).fillna(0)

# 2. Заполнение пропусков в рейтингах
train['product_rating'] = train['product_rating'].fillna(train['product_rating'].median())
test['product_rating'] = test['product_rating'].fillna(test['product_rating'].median())
train['seller_rating'] = train['seller_rating'].fillna(train['seller_rating'].median())
test['seller_rating'] = test['seller_rating'].fillna(test['seller_rating'].median())
train['seller_name'] = train['seller_name'].fillna('unknown')
test['seller_name'] = test['seller_name'].fillna('unknown')

# 3. Создание текстового поля из title + description + highlights
def combine_text(row):
    texts = []
    if pd.notna(row['title']):
        texts.append(str(row['title']))
    if pd.notna(row['description']):
        texts.append(str(row['description']))
    if pd.notna(row['highlights']):
        texts.append(str(row['highlights']))
    return ' '.join(texts)

train['combined_text'] = train.apply(combine_text, axis=1)
test['combined_text'] = test.apply(combine_text, axis=1)

# 4. Заполняем пропуски в ценах медианой
train['selling_price'] = train['selling_price'].fillna(train['selling_price'].median())
train['mrp'] = train['mrp'].fillna(train['mrp'].median())
test['selling_price'] = test['selling_price'].fillna(test['selling_price'].median())
test['mrp'] = test['mrp'].fillna(test['mrp'].median())

# Пересчитываем скидку после заполнения пропусков
train['discount'] = (train['mrp'] - train['selling_price']) / train['mrp']
test['discount'] = (test['mrp'] - test['selling_price']) / test['mrp']
train['discount'] = train['discount'].replace([np.inf, -np.inf], 0).fillna(0)
test['discount'] = test['discount'].replace([np.inf, -np.inf], 0).fillna(0)

# 5. Кодирование целевой переменной
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
train['category_encoded'] = le.fit_transform(train['category_1'])

# 6. Проверяем результат
print("РЕЗУЛЬТАТЫ ОЧИСТКИ:")
print("="*50)
print(f"Пропуски в selling_price: {train['selling_price'].isnull().sum()}")
print(f"Пропуски в mrp: {train['mrp'].isnull().sum()}")
print(f"Пропуски в product_rating: {train['product_rating'].isnull().sum()}")
print(f"Пропуски в seller_rating: {train['seller_rating'].isnull().sum()}")
print(f"Длина combined_text (сред/макс): {train['combined_text'].str.len().mean():.0f} / {train['combined_text'].str.len().max()}")

print(f"\nКодирование классов:")
for i, class_name in enumerate(le.classes_):
    print(f"  {i}: {class_name}")

# 7. Числовые признаки
numeric_features = ['product_rating', 'selling_price', 'mrp', 'discount', 'seller_rating']
print(f"\nЧисловые признаки: {numeric_features}")

РЕЗУЛЬТАТЫ ОЧИСТКИ:
Пропуски в selling_price: 0
Пропуски в mrp: 0
Пропуски в product_rating: 0
Пропуски в seller_rating: 0
Длина combined_text (сред/макс): 393 / 5853

Кодирование классов:
  0: Bady and Kids
  1: Electronics
  2: Home and Furniture
  3: Men's wear
  4: Sports, Books and More
  5: Women's wear

Числовые признаки: ['product_rating', 'selling_price', 'mrp', 'discount', 'seller_rating']


Эксперементы
Цель: найти лучшую комбинацию признаков и модель.
Эксперимент 1: Подбор параметров TF-ID.
Эксперимент 2: Сравнение моделей
Эксперимент 3: Добавление категориального признака seller_name
Эксперимент 4: Кросс-валидация
Stratified 5-fold CV для оценки стабильности модели.
Вывод: средний скор 0.9412, модель стабильна.



In [107]:

numeric_cols = ['product_rating', 'selling_price', 'mrp', 'discount', 'seller_rating']
scaler = StandardScaler()
X_train_num_full = scaler.fit_transform(train[numeric_cols])
X_test_num = scaler.transform(test[numeric_cols])

y = train['category_encoded']

X_train_num, X_val_num, y_train, y_val = train_test_split(
    X_train_num_full, y, test_size=0.2, random_state=42, stratify=y
)


train_idx = y_train.index
val_idx = y_val.index


results = []



tfidf_configs = [
    {'name': 'TF-IDF 5000 unigrams', 'max_features': 5000, 'ngram_range': (1, 1)},
    {'name': 'TF-IDF 5000 unigrams+bigrams', 'max_features': 5000, 'ngram_range': (1, 2)},
    {'name': 'TF-IDF 10000 unigrams+bigrams', 'max_features': 10000, 'ngram_range': (1, 2)},
    {'name': 'TF-IDF 15000 unigrams+bigrams', 'max_features': 15000, 'ngram_range': (1, 2)},
    {'name': 'TF-IDF 5000 unigrams+bigrams+trigrams', 'max_features': 5000, 'ngram_range': (1, 3)},
]

for config in tfidf_configs:
    print(f"\nОбработка: {config['name']}")
    tfidf = TfidfVectorizer(max_features=config['max_features'], ngram_range=config['ngram_range'], min_df=2)
    X_train_text_full = tfidf.fit_transform(train['combined_text'])
    X_train_text = X_train_text_full[train_idx]
    X_val_text = X_train_text_full[val_idx]
    
    X_train_combined = hstack([X_train_text, X_train_num])
    X_val_combined = hstack([X_val_text, X_val_num])
    
  
    lr = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
    lr.fit(X_train_combined, y_train)
    y_pred = lr.predict(X_val_combined)
    score = f1_score(y_val, y_pred, average='macro')
    results.append({
        'model': 'Logistic Regression',
        'features': config['name'],
        'score': score
    })
    print(f"  Logistic Regression: {score:.4f}")

best_tfidf_config = tfidf_configs[1]  
best_score = 0
for r in results:
    if r['score'] > best_score:
        best_score = r['score']
        best_tfidf_config = r['features']

print(f"\nЛучший TF-IDF: {best_tfidf_config} с F1: {best_score:.4f}")



print("\n" + "="*80)
print("ЭКСПЕРИМЕНТЫ С РАЗНЫМИ МОДЕЛЯМИ")
print("="*80)


tfidf_best = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2)
X_train_text_full = tfidf_best.fit_transform(train['combined_text'])
X_train_text = X_train_text_full[train_idx]
X_val_text = X_train_text_full[val_idx]
X_train_combined = hstack([X_train_text, X_train_num])
X_val_combined = hstack([X_val_text, X_val_num])


print("\nRandom Forest:")
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train_combined, y_train)
y_pred = rf.predict(X_val_combined)
score = f1_score(y_val, y_pred, average='macro')
results.append({'model': 'Random Forest', 'features': 'TF-IDF 5000 (1-2)', 'score': score})
print(f"  Score: {score:.4f}")


print("\nGradient Boosting:")
gb = GradientBoostingClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
gb.fit(X_train_combined, y_train)
y_pred = gb.predict(X_val_combined)
score = f1_score(y_val, y_pred, average='macro')
results.append({'model': 'Gradient Boosting', 'features': 'TF-IDF 5000 (1-2)', 'score': score})
print(f"  Score: {score:.4f}")


print("\nXGBoost:")
xgb = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1, eval_metric='mlogloss')
xgb.fit(X_train_combined, y_train)
y_pred = xgb.predict(X_val_combined)
score = f1_score(y_val, y_pred, average='macro')
results.append({'model': 'XGBoost', 'features': 'TF-IDF 5000 (1-2)', 'score': score})
print(f"  Score: {score:.4f}")



print("\n" + "="*80)
print("ЭКСПЕРИМЕНТЫ С ДОБАВЛЕНИЕМ seller_name")
print("="*80)

from sklearn.preprocessing import OneHotEncoder

seller_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
X_train_seller_full = seller_encoder.fit_transform(train['seller_name'].fillna('unknown').values.reshape(-1, 1))
X_train_seller = X_train_seller_full[train_idx]
X_val_seller = X_train_seller_full[val_idx]

X_train_combined_seller = hstack([X_train_combined, X_train_seller])
X_val_combined_seller = hstack([X_val_combined, X_val_seller])

lr_seller = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr_seller.fit(X_train_combined_seller, y_train)
y_pred = lr_seller.predict(X_val_combined_seller)
score = f1_score(y_val, y_pred, average='macro')
results.append({'model': 'Logistic Regression + seller_name', 'features': 'TF-IDF 5000 (1-2)', 'score': score})
print(f"  Score: {score:.4f}")


print("\n" + "="*80)
print("КРОСС-ВАЛИДАЦИЯ (Stratified 5-fold)")
print("="*80)

X_train_full_combined = hstack([X_train_text_full, X_train_num_full])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lr_cv = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
cv_scores = cross_val_score(lr_cv, X_train_full_combined, y, cv=skf, scoring='f1_macro')
print(f"Logistic Regression CV scores: {cv_scores}")
print(f"Mean CV score: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")
results.append({'model': 'Logistic Regression (CV mean)', 'features': 'TF-IDF 5000 (1-2)', 'score': cv_scores.mean()})


print("\n" + "="*80)
print("ИТОГОВАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ")
print("="*80)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('score', ascending=False)
print(results_df.to_string(index=False))

print("\n" + "="*80)
print("ЛУЧШАЯ КОМБИНАЦИЯ:")
print("="*80)
best_row = results_df.iloc[0]
print(f"Модель: {best_row['model']}")
print(f"Признаки: {best_row['features']}")
print(f"F1 Score: {best_row['score']:.4f}")

ЗАПУСК ВСЕХ ЭКСПЕРИМЕНТОВ

Обработка: TF-IDF 5000 unigrams
  Logistic Regression: 0.9390

Обработка: TF-IDF 5000 unigrams+bigrams
  Logistic Regression: 0.9412

Обработка: TF-IDF 10000 unigrams+bigrams
  Logistic Regression: 0.9432

Обработка: TF-IDF 15000 unigrams+bigrams
  Logistic Regression: 0.9425

Обработка: TF-IDF 5000 unigrams+bigrams+trigrams
  Logistic Regression: 0.9410

Лучший TF-IDF: TF-IDF 10000 unigrams+bigrams с F1: 0.9432

ЭКСПЕРИМЕНТЫ С РАЗНЫМИ МОДЕЛЯМИ

Random Forest:
  Score: 0.7880

Gradient Boosting:
  Score: 0.9261

XGBoost:
  Score: 0.9192

ЭКСПЕРИМЕНТЫ С ДОБАВЛЕНИЕМ seller_name
  Score: 0.9412

КРОСС-ВАЛИДАЦИЯ (Stratified 5-fold)
Logistic Regression CV scores: [0.93646007 0.95594179 0.93701803 0.93885026 0.9375162 ]
Mean CV score: 0.9412 (+/- 0.0149)

ИТОГОВАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ
                            model                              features    score
              Logistic Regression         TF-IDF 10000 unigrams+bigrams 0.943224
              Logistic

Обучение модели

In [108]:

numeric_cols = ['product_rating', 'selling_price', 'mrp', 'discount', 'seller_rating']
scaler = StandardScaler()
X_train_num = scaler.fit_transform(train[numeric_cols])
X_test_num = scaler.transform(test[numeric_cols])


tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2)
X_train_text = tfidf.fit_transform(train['combined_text'])
X_test_text = tfidf.transform(test['combined_text'])


seller_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
X_train_seller = seller_encoder.fit_transform(train['seller_name'].fillna('unknown').values.reshape(-1, 1))
X_test_seller = seller_encoder.transform(test['seller_name'].fillna('unknown').values.reshape(-1, 1))


X_train_final = hstack([X_train_text, X_train_num, X_train_seller])
X_test_final = hstack([X_test_text, X_test_num, X_test_seller])

y = train['category_encoded']


lr = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1, C=1.0)
lr.fit(X_train_final, y)
lr_proba = lr.predict_proba(X_test_final)


xgb = XGBClassifier(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
    eval_metric='mlogloss'
)
xgb.fit(X_train_final, y)
xgb_proba = xgb.predict_proba(X_test_final)

blend_proba = (lr_proba + xgb_proba) / 2
y_test_pred = np.argmax(blend_proba, axis=1)
test_predictions = le.inverse_transform(y_test_pred)

submission = pd.DataFrame({
    'id': test['id'],
    'category_1': test_predictions
})

submission.to_csv('КРsubmission.csv', index=False)
print("\nФайл КРsubmission.csv сохранен")
print("\nРаспределение предсказаний:")
print(submission['category_1'].value_counts())


Файл КРsubmission.csv сохранен

Распределение предсказаний:
category_1
Women's wear              493
Men's wear                472
Bady and Kids             460
Home and Furniture        421
Sports, Books and More    374
Electronics               189
Name: count, dtype: int64


Лучший результат на Kaggle: 0.951 при  TF-IDF (max_features=10000, ngram_range=(1,2)) + seller_name + блендинг LogReg и XGBoost
Что я ещё делала: эмбеддинги (ухудшили результат до 0.940), добавление признаков из image_links (дало 0.950)
